In [7]:
from itsai.platform.authentication import DesktopToken
from openai import AzureOpenAI
token_class = DesktopToken()
token_provider = lambda: token_class.token_provider(env="DEV")

client = AzureOpenAI(
    azure_endpoint="https://azapimdev.worldbank.org/conversationalai/v2/",
    api_version="2025-04-01-preview",
    azure_ad_token_provider=token_provider
)
response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": "You are an AI assistant helping with World Bank projects."},
        {"role": "user", "content": "Explain the importance of financial inclusion."}
    ],
    max_completion_tokens=2000,
    reasoning_effort="low"
)
print(response.choices[0].message.content)

Financial inclusion means that individuals and businesses have access to useful, affordable, and appropriate financial products and services — such as payments, savings, credit, insurance, and pensions — delivered in a responsible and sustainable way. Its importance spans economic, social and governance dimensions:

Economic growth and productivity
- Expands investment: Access to credit and formal savings enables households and firms to invest in education, equipment, inputs and technology, raising productivity.
- Supports small businesses: Micro, small and medium enterprises (MSMEs) gain the financing needed to grow, hire, and innovate, which fuels job creation and broader economic activity.
- Improves resource allocation: Formal financial markets channel savings to their most productive uses, increasing overall economic efficiency.

Poverty reduction and shared prosperity
- Smoothes consumption: Access to savings, credit and insurance helps households manage income volatility and sho

In [1]:
import json
import re
import time
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from itsai.platform.authentication import DesktopToken
from openai import AzureOpenAI

INPUT_FILE = Path("acled_kri_enriched.xlsx")
OUTPUT_FILE = Path("acled_kri_enriched_llm_targets.xlsx")
MODEL = "gpt-5-mini"
CHECKPOINT_EVERY = 25
SLEEP_SECONDS = 0
MAX_RETRIES = 3

# Use the same client formatting that is working for you.
token_class = DesktopToken()
token_provider = lambda: token_class.token_provider(env="DEV")

client = AzureOpenAI(
    azure_endpoint="https://azapimdev.worldbank.org/conversationalai/v2/",
    api_version="2025-04-01-preview",
    azure_ad_token_provider=token_provider
)


def find_column(columns, candidates):
    lookup = {str(col).strip().lower(): col for col in columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    raise KeyError(f"Could not find any of these columns: {candidates}")


def build_system_prompt(note_text):
    return f"""
You are classifying one ACLED event note.
Return JSON only with exactly these keys:
- target
- impact_type

Rules:
- Use only the note text provided below.
- Do not add explanation.
- The target list below is ordered from highest risk score to lowest risk score.
- If multiple targets are present, choose the single highest-risk matching target based on that ordered list.
- If none of the targets are present, return "None of these".
- Do not include the score in the output, only the target name.
- For impact_type, return only one of: "Low/Moderate", "Substantial/High", or "Unknown".

Allowed target values ordered from highest score to lowest score:
- Foreign Expatriates [score=5]
- UN AFPs (Not including DPKO) [score=5]
- State Development Agencies [score=5]
- INGO (Staff and Offices) [score=5]
- International Financial Institution (e.g. WBG, IMF, EBRD, ADB, IADB) [score=5]
- Local NGOs [score=4]
- Diplomatic Missions [score=4]
- Critical Civilian (Energy, water, road, train, airport, hospital) [score=4]
- Government (Employees and offices) [score=3]
- Local leaders/Administrators [score=3]
- Students [score=3]
- Political Candidates [score=3]
- UN DPKO Mission [score=3]
- Private Business or Residence [score=3]
- Military (Troops and infrastructure) [score=2]
- Police (Troops and infrastructure) [score=2]
- Information (Radio/TV/Broadcasters) [score=2]
- Vehicle/Transport [score=2]
- Militant Leaders [score=1]
- Militant Troops/Infrastructure [score=1]
- None of these [score=0]

Impact type definitions:
- Low/Moderate: Injuries sustained from events that do not include explosions or weapons.
- Substantial/High: Injuries sustained from events that do include explosions or weapons, or death.
- Unknown: The note does not provide enough information.

Event note:
{note_text}
""".strip()


def parse_response_content(content):
    text = "" if content is None else str(content).strip()
    if not text:
        raise ValueError("Model returned blank content.")

    # Handle fenced markdown JSON if returned.
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{[\s\S]*\}", text)
        if match:
            return json.loads(match.group(0))
        raise ValueError(f"Non-JSON model response: {text[:300]}")


def classify_note(note_text):
    system_prompt = build_system_prompt(note_text)

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": [
                            {
                                "type": "text",
                                "text": system_prompt,
                                "cache_control": {"type": "ephemeral"}
                            }
                        ]
                    },
                    {"role": "user", "content": "Return only valid JSON."}
                ],
                max_completion_tokens=2000,
                reasoning_effort="low"
            )

            parsed = parse_response_content(response.choices[0].message.content)
            return {
                "target": parsed.get("target", "Unknown"),
                "impact_type": parsed.get("impact_type", "Unknown")
            }
        except Exception as exc:
            print(f"Attempt {attempt} failed with error: {exc}")
            if attempt == MAX_RETRIES:
                return {
                    "target": f"ERROR: {exc}",
                    "impact_type": "ERROR"
                }
            time.sleep(2 * attempt)


# Quick preflight to fail fast before full loop.
preflight = classify_note("Armed men attacked a police checkpoint causing injuries.")
print("Preflight result:", preflight)

df = pd.read_excel(INPUT_FILE)
sub_event_col = find_column(df.columns, ["sub_event_type"])
notes_col = find_column(df.columns, ["notes", "note"])

TARGET_COLUMN = "llm_target"
IMPACT_COLUMN = "llm_impact_type"

if TARGET_COLUMN not in df.columns:
    df[TARGET_COLUMN] = "NA"
if IMPACT_COLUMN not in df.columns:
    df[IMPACT_COLUMN] = "NA"

armed_mask = df[sub_event_col].astype(str).str.strip().eq("Armed clash")
armed_indices = df.index[armed_mask].tolist()

# CHECKPOINT RESUMPTION: Load from checkpoint if available
if OUTPUT_FILE.exists():
    print(f"Checkpoint file found: {OUTPUT_FILE.resolve()}")
    df_checkpoint = pd.read_excel(OUTPUT_FILE)
    
    # Find rows that have already been classified (non-"NA" values)
    classified_mask = (df_checkpoint[TARGET_COLUMN] != "NA") & (df_checkpoint[TARGET_COLUMN].notna())
    already_classified = set(df_checkpoint.index[classified_mask].tolist())
    
    # Filter armed_indices to only include unclassified rows
    armed_indices = [idx for idx in armed_indices if idx not in already_classified]
    
    # Use checkpoint data as starting point
    df = df_checkpoint.copy()
    
    print(f"Resuming from checkpoint: {len(already_classified)} already classified")

print(f"Total rows: {len(df)}")
print(f"Armed clash rows to classify: {len(armed_indices)}")
print(f"Using model: {MODEL}")

for counter, row_idx in enumerate(tqdm(armed_indices, desc="Classifying Armed clash events", unit="event"), start=1):
    note_text = str(df.at[row_idx, notes_col]).strip()
    if note_text.lower() == "nan":
        note_text = ""

    result = classify_note(note_text)
    df.at[row_idx, TARGET_COLUMN] = result["target"]
    df.at[row_idx, IMPACT_COLUMN] = result["impact_type"]

    if counter % CHECKPOINT_EVERY == 0:
        df.to_excel(OUTPUT_FILE, index=False)
        print(f"Checkpoint saved after {counter} Armed clash rows")

    time.sleep(SLEEP_SECONDS)

df.to_excel(OUTPUT_FILE, index=False)
print(f"Finished. Output written to: {OUTPUT_FILE.resolve()}")
df[[sub_event_col, notes_col, TARGET_COLUMN, IMPACT_COLUMN]].head(10)

c:\Users\wb649538\projects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Preflight result: {'target': 'Police (Troops and infrastructure)', 'impact_type': 'Substantial/High'}
Checkpoint file found: C:\Users\wb649538\projects\corp_security_demo\acled_kri_enriched_llm_targets.xlsx
Resuming from checkpoint: 3400 already classified
Total rows: 13988
Armed clash rows to classify: 711
Using model: gpt-5-mini


Classifying Armed clash events:   4%|▎         | 25/711 [02:32<2:37:32, 13.78s/event]

Checkpoint saved after 25 Armed clash rows


Classifying Armed clash events:   7%|▋         | 50/711 [05:18<2:13:20, 12.10s/event]

...

Classifying Armed clash events:  98%|█████████▊| 700/711 [1:16:24<01:50, 10.03s/event]

Checkpoint saved after 700 Armed clash rows


Classifying Armed clash events: 100%|██████████| 711/711 [1:17:35<00:00,  6.55s/event]


Finished. Output written to: C:\Users\wb649538\projects\corp_security_demo\acled_kri_enriched_llm_targets.xlsx


,sub_event_type,notes,llm_target,llm_impact_type
0,Armed clash,"On 1 January 2024, an unidentified armed group...",Government (Employees and offices),Substantial/High
1,Attack,"On 1 January 2024, suspected Boko Haram [JAS f...",NaN,NaN
2,Attack,"On 1 January 2024, suspected Boko Haram [JAS f...",NaN,NaN
3,Abduction/forced disappearance,"On 1 January 2024, a Zamfara militia abducted ...",NaN,NaN
4,Attack,"On 1 January 2024, an unidentified armed group...",NaN,NaN
5,Peaceful protest,"On 1 January 2024, a group of women protested ...",NaN,NaN
6,Peaceful protest,"On 1 January 2024, land owners in Igbesa town ...",NaN,NaN
7,Attack,"On 1 January 2024, an unidentified armed group...",NaN,NaN
8,Attack,"On 1 January 2024, an unidentified armed group...",NaN,NaN
9,Armed clash,"Around 2 January 2024 (as reported), a soldier...",Military (Troops and infrastructure),Substantial/High
